# DeepLog Language

A quick tour of the textual DeepLog language, how it maps to the parser and grammar, and how to compile formulas into modules.


## Overview

DeepLog provides a compact textual language for writing formulas with aggregations, transformations, unary/binary operators, and leaves tagged with structures (e.g., `_boolean`, `_probability`, `_logprobability`).

Parser helpers:

- {func}`~deeplog.formula.text_parser_lark.parse_formula` to view the symbolic tuple representation that matches the grammar.
- {func}`~deeplog.formula.text_parser_lark.parse_formula_to_module` to execute the same text as a module.


In [ ]:
import torch

from deeplog import parse_formula_to_module
from deeplog.formula import parse_formula, SymbolicFormulaFactory

## Leaves and structures

Leaves are written as `predicate(args)_structure`. Structures tag the "domain" of the atom (e.g., `_boolean`, `_probability`, `_logprobability`).


In [ ]:
text = "=(Burglary,true)_boolean"
parsed = parse_formula(text, SymbolicFormulaFactory())
parsed

## Aggregations, binary, and unary operators

- Aggregation: `agg(op, Var): expression` binds `Var` in `expression`.
- Binary: `lhs op rhs` (all binary ops share the same precedence and associate left).
- Unary: `op expr` applies a prefix operator.
- Parentheses + `_structure` create transformations: `(expr)_probability`.


In [ ]:
model_count_text = """
sum(Burglary): sum(Earthquake):
    =(Burglary,true)_boolean or =(Earthquake,true)_boolean
"""
parsed_model_count = parse_formula(model_count_text, SymbolicFormulaFactory())
parsed_model_count

You can also compile the same string directly into a module. The resulting {class}`~deeplog.module.deeplog_module.DeepLogModule` validates shapes and plugs into Torch code.


In [ ]:
model_count_module = parse_formula_to_module(model_count_text)
# Evaluate on a dummy batch with no free variables (shape: batch x 0)
out = model_count_module()
out

## Mixing structures: probabilities over Boolean formulas

Transformations let you change structures mid-formula (e.g., turn a Boolean result into a probability and multiply by literal weights).


In [ ]:
weighted_text = """
sum(Burglary): sum(Earthquake):
    ((=(Burglary,true)_boolean or =(Earthquake,true)_boolean)_probability)
    times
    (p(Burglary)_probability times p(Earthquake)_probability)
"""
parse_formula(weighted_text, SymbolicFormulaFactory())

In [ ]:
from deeplog import reshape
from deeplog.shape import SymTensor

weighted_module = parse_formula_to_module(weighted_text)
print("Original input shape:", weighted_module.get_input_shape())

# Fix input ordering: Burglary first, then Earthquake
expected_input = SymTensor([
    ("_", ("p", ("Burglary",)), ("probability",)),
    ("_", ("p", ("Earthquake",)), ("probability",)),
])
weighted_module = reshape(weighted_module, input=expected_input)

# Evaluate with P(Burglary) = 0.2 and P(Earthquake) = 0.6
weighted_module(torch.tensor([[0.2, 0.6]]))


## Tips

- All binary operators share the same precedence; use parentheses to force grouping.
- Aggregations capture the following expression before binary/unary operators are applied.
- Comments starting with `#` and extra whitespace are ignored.
- Leaf suffixes `_boolean`, `_probability`, `_logprobability` pick the structure of the atom.
- Use `parse_formula` when you need the symbolic tuple; use `parse_formula_to_module` when you want a runnable module.
